Import Libraries

In [38]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
import joblib

Load the Dataset using the directrory of the stored, df.head() shows some rows in the header part of the dataset

In [39]:

df = pd.read_csv('../../data sheets/synthetic_face_shape_dataset_5000.csv') 
df.head()

,forehead width (cm),jaw width (cm),face length (cm),face width (cm),face shape,best hairstyle
0,13.702252,12.724586,17.306009,14.490913,Square,Soft Curls
1,13.664632,13.359635,18.084700,13.225752,Diamond,Tousled Waves
2,14.718268,11.072230,18.924932,13.756551,Heart,Chin-Length Bob
3,14.194716,10.229002,18.447512,13.390788,Oval,Side Swept Bangs
4,14.674758,13.282522,18.614628,13.971958,Oblong,Curtain Bangs


Define Features and Target as X: inputs(the features used to predict) and y : Output result (the label that want to predict = face shape)

In [40]:
X = df[['face width (cm)', 'jaw width (cm)', 'forehead width (cm)']]
y = df['face shape']

split the data in to training and testing data sets (80% for training, 20% for testing)and the random_state=42 ensures the split is reproducible.

In [41]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

 Scale the Features,Standardizes values like face width so all numbers are on the same scale.

In [42]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Tune and Train Logistic Regression Using Grid Search( Grid SearchCV automatically picks the best model with the best accuracy)

In [43]:
param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'solver': ['lbfgs', 'liblinear'],
    'penalty': ['l2']
}

grid_search = GridSearchCV(
    LogisticRegression(max_iter=1000),
    param_grid,
    cv=5,
    scoring='accuracy',
    verbose=1,
    n_jobs=-1
)

grid_search.fit(X_train_scaled, y_train)
logreg_model = grid_search.best_estimator_

print("Best Parameters:", grid_search.best_params_)


Fitting 5 folds for each of 8 candidates, totalling 40 fits


/root/miniforge3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1288: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
/root/miniforge3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1288: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
/root/miniforge3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1288: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports 

Best Parameters: {'C': 10, 'penalty': 'l2', 'solver': 'lbfgs'}


/root/miniforge3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1288: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
/root/miniforge3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1288: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(


Evaluate the Model (Predicts face shape using the test set,and also shows the accuracy of the system that means, how well the model performed) here the accuracy is 68.9% for the 6 classes it is better but it have to imtrove like adding more features.

In [44]:
y_pred = logreg_model.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.689
              precision    recall  f1-score   support

     Diamond       0.59      0.66      0.62       137
       Heart       0.83      0.82      0.83       148
      Oblong       0.70      0.83      0.76       191
        Oval       0.88      0.86      0.87       162
       Round       0.44      0.36      0.40       175
      Square       0.66      0.62      0.64       187

    accuracy                           0.69      1000
   macro avg       0.69      0.69      0.69      1000
weighted avg       0.68      0.69      0.68      1000



Create Rule-Based Hairstyle Recommender(it suggest a hairstyle without training a second model, only use rule based)

In [45]:
face_to_hairstyle = (
    df.drop_duplicates(subset=['face shape'])
      .set_index('face shape')['best hairstyle']
      .to_dict()
)

Predict Function for New Inputs, by recalling for the new data

In [46]:
def predict_face_shape_and_hairstyle(input_features):
    input_df = pd.DataFrame([input_features], columns=['face width (cm)', 'jaw width (cm)', 'forehead width (cm)'])
    scaled_input = scaler.transform(input_df)
    predicted_shape = logreg_model.predict(scaled_input)[0]
    recommended_hairstyle = face_to_hairstyle.get(predicted_shape, "No suggestion")
    return predicted_shape, recommended_hairstyle

Test It With Example Measurements

In [47]:
new_face = [150, 130, 153]  # Example values
shape, hairstyle = predict_face_shape_and_hairstyle(new_face)

print(f"Predicted Face Shape: {shape}")
print(f"Recommended Hairstyle: {hairstyle}")

Predicted Face Shape: Square
Recommended Hairstyle: Soft Curls
